In [0]:
%sql
-- ΕΝΟΤΗΤΑ 1: Αρχικοποίηση Περιβάλλοντος και Δημιουργία Σχημάτων (Medallion Setup)
CREATE WIDGET TEXT catalog DEFAULT "bootcamp";
CREATE WIDGET TEXT bronze_schema DEFAULT "bronze_marios_urban";
CREATE WIDGET TEXT silver_schema DEFAULT "silver_marios_urban";
CREATE WIDGET TEXT gold_schema DEFAULT "gold_marios_urban";
CREATE WIDGET TEXT volume_landing_path DEFAULT "/Volumes/bootcamp/bronze_marios_urban/landing";

In [0]:
%sql
USE SCHEMA IDENTIFIER(:bronze_schema);
CREATE VOLUME IF NOT EXISTS landing;

In [0]:
%sql
-- Βήμα 2.1: Bronze Ingestion — customers.csv -> brz_customers. 
-- Φόρτωση του customers.csv σε Delta table (brz_customers) με όλες τις στήλες STRING 
-- για ασφαλές ingestion, συν τεχνικές στήλες lineage (_source_file, _source_row_hash, _ingested_at) 
-- για ιχνηλασιμότητα και εντοπισμό διπλότυπων.
USE SCHEMA IDENTIFIER(:bronze_schema);
CREATE OR REPLACE TABLE brz_customers AS 
SELECT 
customer_id,
first_name,
last_name,
email,
gender,
birth_date,
city,
region,
segment,
signup_date,
_metadata.file_path AS _source_file, 
sha2(
    concat_ws(
        '||',
         trim(customer_id),
         trim(first_name), 
         trim(last_name), 
         trim(email), 
         trim(gender), 
         trim(birth_date), 
         trim(city), 
         trim(region), 
         trim(segment), 
         trim(signup_date)
         ), 
         256
         ) AS _source_row_hash,
current_timestamp() AS _ingested_at
FROM read_files('/Volumes/bootcamp/bronze_marios_urban/landing/customers.csv',format => 'csv',
  header => true,
  schema => 'customer_id STRING, first_name STRING, last_name STRING, email STRING, gender STRING, birth_date STRING, city STRING, region STRING, segment STRING, signup_date STRING'); 


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) FROM brz_customers;

count(*)
508


In [0]:
%sql
-- Βήμα 2.2: Bronze Ingestion — customer_changes.csv -> brz_customer_changes

CREATE OR REPLACE TABLE brz_customer_changes AS 
SELECT 
customer_id,
new_city,
new_region,
new_segment,
effective_date,
_metadata.file_path AS _source_file,
sha2(
    concat_ws(
        '||',
         trim(customer_id), 
         trim(new_city), 
         trim(new_region), 
         trim(new_segment), 
         trim(effective_date)
         ),  
         256
        ) AS _source_row_hash,
current_timestamp() AS _ingested_at
FROM read_files('/Volumes/bootcamp/bronze_marios_urban/landing/customer_changes.csv',format => 'csv',
  header => true,
  schema => 'customer_id STRING, new_city STRING, new_region STRING, new_segment STRING, effective_date STRING');

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Βήμα 2.3: Bronze Ingestion — products.csv -> brz_products
CREATE OR REPLACE TABLE brz_products AS 
SELECT 
product_id,
product_name,
category,
subcategory,
brand,
unit_cost,
list_price,
is_active,
_metadata.file_path AS _source_file,
sha2(
    concat_ws(
        '||',
         trim(product_id), 
         trim(product_name), 
         trim(category), 
         trim(subcategory), 
         trim(brand),
         trim(unit_cost),
         trim(list_price),
         trim(is_active)
         ),  
         256
        ) AS _source_row_hash,
current_timestamp() AS _ingested_at
FROM read_files('/Volumes/bootcamp/bronze_marios_urban/landing/products.csv',format => 'csv',
  header => true,
  schema => 'product_id STRING, product_name STRING, category STRING, subcategory STRING, brand STRING, unit_cost STRING, list_price STRING, is_active STRING');

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Βήμα 2.4: Bronze Ingestion — stores.csv -> brz_stores
CREATE OR REPLACE TABLE brz_stores AS
SELECT
  store_id,
  store_name,
  store_type,
  city,
  region,
  open_date,
  size_sqm,
  _metadata.file_path AS _source_file,
  sha2(
    concat_ws(
      '||',
      trim(store_id),
      trim(store_name),
      trim(store_type),
      trim(city),
      trim(region),
      trim(open_date),
      trim(size_sqm)
    ),
    256
  ) AS _source_row_hash,
  current_timestamp() AS _ingested_at
FROM read_files(
  '/Volumes/bootcamp/bronze_marios_urban/landing/stores.csv',
  format => 'csv',
  header => true,
  schema => 'store_id STRING, store_name STRING,store_type STRING ,city STRING, region STRING, open_date STRING,size_sqm STRING'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
--Βήμα 2.5: Bronze Ingestion — orders.csv -> brz_orders
CREATE OR REPLACE TABLE brz_orders AS
SELECT
  order_id,
  order_ts,
  customer_id,
  store_id,
  payment_method,
  order_status,
  _metadata.file_path AS _source_file,
  sha2(
    concat_ws(
      '||',
      trim(order_id),
      trim(order_ts),
      trim(customer_id),
      trim(store_id),
      trim(payment_method),
      trim(order_status)
    ),
    256
  ) AS _source_row_hash,
  current_timestamp() AS _ingested_at
FROM read_files(
  '/Volumes/bootcamp/bronze_marios_urban/landing/orders.csv',
  format => 'csv',
  header => true,
  schema => 'order_id STRING, order_ts STRING, customer_id STRING, store_id STRING, payment_method STRING, order_status STRING'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
--Βήμα 2.6: Bronze Ingestion — order_items.csv -> brz_order_items
CREATE OR REPLACE TABLE brz_order_items AS
SELECT
  order_line_id,
  order_id,
  product_id,
  quantity,
  unit_price,
  discount_pct,
  _metadata.file_path AS _source_file,
  sha2(
    concat_ws(
      '||',
      trim(order_line_id),
      trim(order_id),
      trim(product_id),
      trim(quantity),
      trim(unit_price),
      trim(discount_pct)
    ),
    256
  ) AS _source_row_hash,
  current_timestamp() AS _ingested_at
FROM read_files(
  '/Volumes/bootcamp/bronze_marios_urban/landing/order_items.csv',
  format => 'csv',
  header => true,
  schema => 'order_line_id STRING, order_id STRING, product_id STRING, quantity STRING, unit_price STRING, discount_pct STRING'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Βήμα 2.7: Bronze Layer Validation & Row Counts

SELECT 'brz_customers' AS table_name, count(*) AS row_count FROM brz_customers
UNION ALL
SELECT 'brz_customer_changes' AS table_name, count(*) AS row_count FROM brz_customer_changes
UNION ALL
SELECT 'brz_products' AS table_name, count(*) AS row_count FROM brz_products
UNION ALL
SELECT 'brz_stores' AS table_name, count(*) AS row_count FROM brz_stores
UNION ALL
SELECT 'brz_orders' AS table_name, count(*) AS row_count FROM brz_orders
UNION ALL
SELECT 'brz_order_items' AS table_name, count(*) AS row_count FROM brz_order_items;

table_name,row_count
brz_customers,508
brz_customer_changes,25
brz_products,76
brz_stores,13
brz_orders,8000
brz_order_items,24111


In [0]:
%sql
/*
  Phase 1 · Deliverable: Data Quality (DQ) Log
  Συγκεντρωτική καταγραφή προβλημάτων ποιότητας δεδομένων, πλήθους γραμμών και κανόνων διαχείρισης.
*/
SELECT 
  source_table,
  reject_reason AS issue_identified,
  count(*) AS row_count,
  CASE 
    WHEN lower(reject_reason) LIKE '%duplicate%' 
      THEN 'Deduplicate by customer_id; quarantine duplicate records to etl_rejects'
    WHEN lower(reject_reason) LIKE '%email%' OR lower(reject_reason) LIKE '%null%' 
      THEN 'Filter out records with missing email; quarantine to etl_rejects'
    WHEN lower(reject_reason) LIKE '%quantity%' OR lower(reject_reason) LIKE '%<= 0%' 
      THEN 'Filter out non-positive quantities; quarantine to etl_rejects'
    WHEN lower(reject_reason) LIKE '%orphan%' OR lower(reject_reason) LIKE '%foreign key%' 
      THEN 'Quarantine referential integrity violations to etl_rejects'
    WHEN lower(reject_reason) LIKE '%invalid product%' OR lower(reject_reason) LIKE '%p9999%' 
      THEN 'Quarantine to etl_rejects and map to Unknown Member (-1) in Gold'
    ELSE 'Quarantine to silver.etl_rejects'
  END AS handling_rule
FROM bootcamp.silver_marios_urban.etl_rejects
GROUP BY source_table, reject_reason
ORDER BY source_table, row_count DESC;

source_table,issue_identified,row_count,handling_rule
brz_customers,MISSING_EMAIL,15,Filter out records with missing email; quarantine to etl_rejects
brz_customers,DUPLICATE_CUSTOMER,8,Deduplicate by customer_id; quarantine duplicate records to etl_rejects
brz_order_items,INVALID_QUANTITY,113,Filter out non-positive quantities; quarantine to etl_rejects
brz_products,INVALID_PRICE_OR_CATEGORY,1,Quarantine to silver.etl_rejects


# Phase 2 — Deliverable: Dimensional Design Note & ERD

## 1. Kimball's 4 Steps

* **Step 1: Business Process:** Μοντελοποίηση λιανικών πωλήσεων (Retail Sales Orders) της UrbanMart σε φυσικά καταστήματα και online[cite: 9].
* **Step 2: Fact Grain:** One row in the fact table represents an individual line item within a customer sales order (μία γραμμή ανά προϊόν παραγγελίας)[cite: 9].
* **Step 3: Dimensions:**
  * `dim_customers`: Στοιχεία πελατών με παρακολούθηση ιστορικότητας ή τρέχουσας κατάστασης (`customer_id`, ονοματεπώνυμο, email, δημογραφικά)[cite: 9].
  * `dim_products`: Κατάλογος προϊόντων με κατηγοριοποίηση (`product_id`, `brand`, `category`, `subcategory`) και τιμολόγηση[cite: 9].
  * `dim_stores`: Φυσικά καταστήματα και γεωγραφικά στοιχεία (`store_id`, `store_type`, `city`, `region`, `size_sqm`)[cite: 9].
  * `dim_date`: Ημερολογιακή διάσταση για χρονικές αναλύσεις (`date_key`, `year`, `quarter`, `month`, `day_of_week`, `is_weekend`)[cite: 9].
* **Step 4: Fact Table (`fact_sales`):**
  * **Primary Key:** `sales_key` (surrogate key)[cite: 9]
  * **Foreign Keys:** `customer_key`, `product_key`, `store_key`, `order_date_key`[cite: 9]
  * **Degenerate Dimensions:** `order_id`, `order_line_id`, `payment_method`, `order_status`[cite: 9]

---

## 2. Classification of Measures

* **Additive (Πλήρως Αθροίσιμα με SUM σε όλες τις διαστάσεις):**
  * `quantity`: Πλήθος τεμαχίων[cite: 9]
  * `gross_amount`: Ακαθάριστος τζίρος (`quantity * unit_price`)[cite: 9]
  * `discount_amount`: Ποσό έκπτωσης (`quantity * unit_price * discount_pct`)[cite: 9]
  * `net_amount`: Καθαρός τζίρος (`gross_amount - discount_amount`)[cite: 9]
  * `cost_amount`: Συνολικό κόστος (`quantity * cost_price`)[cite: 7, 9]
  * `margin_amount`: Καθαρό κέρδος (`net_amount - cost_amount`)[cite: 7, 9]
* **Semi-additive:** Κανένα (δεν υπάρχουν μετρήσεις υπολοίπων / snapshot ισολογισμού)[cite: 9].
* **Non-additive (Μη Αθροίσιμα — τιμές αναφοράς και ποσοστά):**
  * `unit_price`: Τιμή πώλησης ανά μονάδα[cite: 9]
  * `cost_price`: Κόστος ανά μονάδα[cite: 9]
  * `discount_pct`: Ποσοστό έκπτωσης[cite: 9]

---

## 3. Handling of Unknown Members & Guest Orders

* **Μηχανισμός:** Χρήση ειδικής τεχνικής εγγραφής **Unknown Member με surrogate key `-1`** στις διαστάσεις (`dim_customers`, `dim_products`, `dim_stores`)[cite: 7].
* **Επιχειρηματική Αιτιολόγηση:** Συναλλαγές επισκεπτών χωρίς στοιχεία πελάτη (`customer_id IS NULL`) ή παραγγελίες με μη καταγεγραμμένα/άκυρα προϊόντα δεν απορρίπτονται, αλλά αντιστοιχίζονται στο κλειδί `-1`[cite: 7]. Έτσι διασφαλίζεται ότι ο πραγματικός τζίρος και οι πωλήσεις διατηρούνται ακέραιες στον πίνακα γεγονότων (`fact_sales`) χωρίς να παραβιάζεται η αναφορική ακεραιότητα[cite: 7].

---

## 4. Entity-Relationship Diagram (ERD — Star Schema)

```text
       +-----------------------+
       |     dim_customers     |
       +-----------------------+
       | PK  customer_key      |<---------+
       |     customer_id       |          |
       |     first_name        |          |
       |     last_name         |          |
       |     email             |          |
       |     city, region      |          |
       +-----------------------+          |
                                          |
       +-----------------------+          |          +-----------------------+
       |      dim_stores       |          |          |     dim_products      |
       +-----------------------+          |          +-----------------------+
       | PK  store_key         |<------+  |  +------>| PK  product_key       |
       |     store_id          |       |  |  |       |     product_id        |
       |     store_name        |       |  |  |       |     product_name      |
       |     store_type        |       |  |  |       |     category, brand   |
       |     city, region      |       |  |  |       |     cost/list_price   |
       +-----------------------+       |  |  |       +-----------------------+
                                       |  |  |
                                       |  |  |
                        +--------------+--+--+--------------+
                        |                fact_sales         |
                        +-----------------------------------+
                        | PK  sales_key                     |
                        | FK  order_date_key                |--------+
                        | FK  customer_key                  |        |
                        | FK  product_key                   |        |
                        | FK  store_key                     |        |
                        |     order_id (DD)                 |        |
                        |     order_line_id (DD)            |        |
                        |     payment_method (DD)           |        |
                        |     order_status (DD)             |        |
                        |     quantity                      |        |
                        |     unit_price                    |        |
                        |     discount_pct                  |        |
                        |     gross_amount                  |        |
                        |     discount_amount               |        |
                        |     net_amount                    |        |
                        |     cost_amount                   |        |
                        |     margin_amount                 |        |
                        +-----------------------------------+        |
                                                                     |
                                                                     |
                                      +-----------------------+      |
                                      |       dim_date        |      |
                                      +-----------------------+      |
                                      | PK  date_key          |<-----+
                                      |     date              |
                                      |     year, quarter     |
                                      |     month, month_name |
                                      |     day_of_month      |
                                      |     day_name          |
                                      |     is_weekend        |
                                      +-----------------------+